In [1]:
# Scenario: Corporate Market Research & Strategy
# A company wants to explore launching a new product in a competitive market.
# The Manager Agent oversees the process and delegates tasks to specialized workers.

# ================================
# MANAGER AGENT
# - Receives the overall goal
# - Dynamically assigns tasks to worker agents depending on what is needed
# ================================

# ================================
# WORKER AGENTS
# - Market Research Worker
# - Finance Worker
# - Operations Worker
# - Legal Worker
# - HR Worker
# ================================

import json
import re
from typing import Dict, Any
from groq import Groq


GROQ_API_KEY = "YOUR_NEW_GROQ_API_KEY_HERE"
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found")

client = Groq(api_key=GROQ_API_KEY)


def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_completion_tokens=1400
    )

    content = completion.choices[0].message.content
    if not content or not content.strip():
        raise ValueError("Empty response returned by model")

    return content.strip()


def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No valid JSON object found. Raw response: {text}")

        cleaned = match.group(0)
        cleaned = cleaned.replace("\r", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = cleaned.replace("\n", " ")

        try:
            return json.loads(cleaned)
        except Exception:
            cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            return json.loads(cleaned)


class ManagerAgent:
    def create_plan(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("[Manager Agent] Reviewing strategic objective and assigning worker tasks...")

        system_prompt = (
            "You are the Manager Agent in a corporate market research and strategy multi-agent system. "
            "Your role is dynamic orchestration and task delegation. "
            "You receive the overall business goal and product context. "
            "You must decide which worker agents should be activated based on the strategic need. "
            "Available workers are: Market Research Worker, Finance Worker, Operations Worker, Legal Worker, HR Worker. "
            "If market demand, competition, segmentation, or customer behavior matters, assign Market Research Worker. "
            "If budget, pricing, investment, ROI, or financial feasibility matters, assign Finance Worker. "
            "If supply chain, logistics, production capacity, or operational execution matters, assign Operations Worker. "
            "If regulations, certifications, intellectual property, contracts, import rules, or compliance matter, assign Legal Worker. "
            "If staffing, hiring, workforce readiness, or training matters, assign HR Worker. "
            "Return only valid JSON with exactly these keys: "
            "goal_summary, assigned_workers, planning_rationale, manager_notes. "
            "Rules: "
            "goal_summary must explain the strategic objective clearly in 2 to 4 lines; "
            "assigned_workers must be a list containing worker names exactly as defined; "
            "planning_rationale must explain why those workers are needed in 4 to 6 lines; "
            "manager_notes must provide concise execution guidance for the downstream workflow. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Manager Agent.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Understand the business objective clearly.
2. Decide which worker agents are required.
3. Build a practical delegation plan.
4. Make sure the plan covers the most important launch feasibility dimensions.
5. Keep the answer executive-friendly and operationally useful.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)

    def final_decision(self, goal: str, product_data: Dict[str, Any], worker_reports: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Manager Agent] Consolidating worker reports into final strategic recommendation...")

        system_prompt = (
            "You are the Manager Agent in a corporate market research and strategy multi-agent system. "
            "Your role now is final synthesis and strategic decision-making. "
            "You receive the overall goal, product context, and all worker reports. "
            "You must integrate market, finance, operations, legal, and HR insights into one final recommendation about product launch feasibility. "
            "Your output should feel like a polished management review summary. "
            "Return only valid JSON with exactly these keys: "
            "launch_feasibility, key_strengths, key_risks, final_recommendation, execution_strategy. "
            "Rules: "
            "launch_feasibility must be one of Not Feasible, Feasible with Conditions, or Highly Feasible; "
            "key_strengths must be a short list of the strongest positive factors; "
            "key_risks must be a short list of the major concerns or launch blockers; "
            "final_recommendation must be 4 to 7 lines and clearly state whether the product should launch and under what conditions; "
            "execution_strategy must explain how the company should move forward in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Manager Agent for final decision-making.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Worker Reports:
{json.dumps(worker_reports, indent=2)}

Requirements:
1. Review all worker reports together.
2. Assess whether launch is feasible.
3. Highlight the strongest opportunities.
4. Highlight the biggest risks.
5. Provide a practical and well-worded final recommendation.
6. Suggest a realistic execution strategy.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class MarketResearchWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Market Research Worker] Collecting competitor data, customer preferences, and demand forecasts...")

        system_prompt = (
            "You are the Market Research Worker in a corporate strategy multi-agent system. "
            "Your role is market intelligence only. "
            "You must evaluate customer demand, market attractiveness, competition intensity, and customer expectations for a new product launch. "
            "Think like a senior market research analyst. "
            "Return only valid JSON with exactly these keys: "
            "demand_forecast, competition_level, customer_preferences, market_insight. "
            "Rules: "
            "demand_forecast must summarize expected demand clearly; "
            "competition_level must describe the competitive environment; "
            "customer_preferences must be a short list of key customer preferences; "
            "market_insight must explain the market opportunity in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Market Research Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Estimate demand direction.
2. Assess competition level.
3. Identify important customer preferences.
4. Provide a practical market insight for launch feasibility.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class FinanceWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Finance Worker] Analyzing budget, ROI, and pricing strategy...")

        system_prompt = (
            "You are the Finance Worker in a corporate strategy multi-agent system. "
            "Your role is financial feasibility only. "
            "You must evaluate likely investment, ROI timeline, pricing approach, and financial risk for the product launch. "
            "Think like a finance strategy lead preparing a launch feasibility memo. "
            "Return only valid JSON with exactly these keys: "
            "estimated_investment, expected_roi_period, pricing_strategy, financial_risk, financial_insight. "
            "Rules: "
            "estimated_investment must provide a realistic high-level investment estimate; "
            "expected_roi_period must estimate likely return period; "
            "pricing_strategy must describe a sensible pricing approach; "
            "financial_risk must be Low, Medium, or High; "
            "financial_insight must explain whether the launch is financially attractive in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Finance Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Estimate the investment scale.
2. Estimate the likely ROI horizon.
3. Suggest an appropriate pricing strategy.
4. Assess the overall financial risk.
5. Provide a concise finance insight.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class OperationsWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Operations Worker] Evaluating production capacity, supply chain, and logistics...")

        system_prompt = (
            "You are the Operations Worker in a corporate strategy multi-agent system. "
            "Your role is operational feasibility only. "
            "You must evaluate production readiness, supply chain stability, logistics practicality, and operational scale-up capability. "
            "Think like a senior operations strategist. "
            "Return only valid JSON with exactly these keys: "
            "production_capacity, supply_chain_status, logistics_challenge, operations_insight. "
            "Rules: "
            "production_capacity must describe whether the business can scale supply; "
            "supply_chain_status must summarize sourcing and operational stability; "
            "logistics_challenge must identify the main execution difficulty; "
            "operations_insight must explain operational launch readiness in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Operations Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Evaluate operational readiness.
2. Review supply chain status.
3. Identify the main logistics challenge.
4. Provide a practical operations insight.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class LegalWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Legal Worker] Reviewing compliance, intellectual property, and regional regulations...")

        system_prompt = (
            "You are the Legal Worker in a corporate strategy multi-agent system. "
            "Your role is legal and regulatory feasibility only. "
            "You must assess trademark availability, regulatory obligations, certification needs, import or market-entry restrictions, and compliance risk. "
            "Think like senior legal counsel supporting market entry strategy. "
            "Return only valid JSON with exactly these keys: "
            "trademark_status, compliance_status, legal_risk, legal_insight. "
            "Rules: "
            "trademark_status must indicate whether the intellectual property path appears clear; "
            "compliance_status must summarize regulatory or certification requirements; "
            "legal_risk must be Low, Medium, or High; "
            "legal_insight must explain launch readiness from a legal standpoint in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Legal Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Review trademark or IP readiness.
2. Identify compliance and regulatory obligations.
3. Assess legal risk.
4. Provide a concise legal insight for launch readiness.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class HRWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[HR Worker] Assessing staffing needs and training requirements...")

        system_prompt = (
            "You are the HR Worker in a corporate strategy multi-agent system. "
            "Your role is workforce readiness only. "
            "You must evaluate staffing scale, hiring priorities, and training readiness needed to support a product launch. "
            "Think like an HR planning leader supporting expansion. "
            "Return only valid JSON with exactly these keys: "
            "new_hires_required, roles_needed, training_need, hr_insight. "
            "Rules: "
            "new_hires_required must estimate likely hiring scale; "
            "roles_needed must list the most important roles; "
            "training_need must describe key training requirements; "
            "hr_insight must explain whether workforce readiness supports launch in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the HR Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Estimate hiring requirements.
2. Identify critical roles needed for launch.
3. Assess training needs.
4. Provide a concise HR readiness insight.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class CorporateStrategySystem:
    def __init__(self):
        self.manager = ManagerAgent()
        self.market_worker = MarketResearchWorker()
        self.finance_worker = FinanceWorker()
        self.operations_worker = OperationsWorker()
        self.legal_worker = LegalWorker()
        self.hr_worker = HRWorker()

    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        manager_plan = self.manager.create_plan(goal, product_data)
        assigned_workers = manager_plan.get("assigned_workers", [])

        worker_reports = {}

        for worker_name in assigned_workers:
            if worker_name == "Market Research Worker":
                worker_reports["market_research_report"] = self.market_worker.run(goal, product_data)

            elif worker_name == "Finance Worker":
                worker_reports["finance_report"] = self.finance_worker.run(goal, product_data)

            elif worker_name == "Operations Worker":
                worker_reports["operations_report"] = self.operations_worker.run(goal, product_data)

            elif worker_name == "Legal Worker":
                worker_reports["legal_report"] = self.legal_worker.run(goal, product_data)

            elif worker_name == "HR Worker":
                worker_reports["hr_report"] = self.hr_worker.run(goal, product_data)

        final_decision = self.manager.final_decision(goal, product_data, worker_reports)

        return {
            "manager_plan": manager_plan,
            "worker_reports": worker_reports,
            "manager_final_decision": final_decision
        }


goal = "Evaluate feasibility of launching Product Y in Asia"

product_data = {
    "product_name": "Product Y",
    "target_region": "Asia",
    "industry": "Consumer Technology",
    "budget_unclear": True,
    "regulations_complex": True,
    "production_ready": True,
    "staffing_expansion_needed": True,
    "target_customers": "Urban and digitally active middle-income consumers",
    "launch_objective": "Establish strong presence in high-demand Asian metro markets first"
}

system = CorporateStrategySystem()
final_output = system.run(goal, product_data)

print("\n" + "=" * 72)
print("CORPORATE MARKET RESEARCH & STRATEGY OUTPUT")
print("=" * 72)

print("\n1. Manager Planning Output")
print(json.dumps(final_output["manager_plan"], indent=2))

if "market_research_report" in final_output["worker_reports"]:
    print("\n2. Market Research Worker Output")
    print(json.dumps(final_output["worker_reports"]["market_research_report"], indent=2))

if "finance_report" in final_output["worker_reports"]:
    print("\n3. Finance Worker Output")
    print(json.dumps(final_output["worker_reports"]["finance_report"], indent=2))

if "operations_report" in final_output["worker_reports"]:
    print("\n4. Operations Worker Output")
    print(json.dumps(final_output["worker_reports"]["operations_report"], indent=2))

if "legal_report" in final_output["worker_reports"]:
    print("\n5. Legal Worker Output")
    print(json.dumps(final_output["worker_reports"]["legal_report"], indent=2))

if "hr_report" in final_output["worker_reports"]:
    print("\n6. HR Worker Output")
    print(json.dumps(final_output["worker_reports"]["hr_report"], indent=2))

print("\n7. Manager Final Decision")
print(json.dumps(final_output["manager_final_decision"], indent=2))

print("\n" + "=" * 72)
print("FINAL SYSTEM SUMMARY")
print("=" * 72)
print("""
This system demonstrates a manager-worker multi-agent architecture for corporate market research and launch strategy evaluation.

At the center of the workflow, the Manager Agent receives the strategic objective and dynamically decides which worker agents should be activated. This makes the system adaptive rather than rigid, because delegation changes based on business context such as budget uncertainty, regulatory complexity, operational readiness, and staffing needs.

Once the plan is created, specialist workers operate in their own domains.
The Market Research Worker studies demand, competition, and customer preferences.
The Finance Worker evaluates investment logic, ROI outlook, and pricing direction.
The Operations Worker reviews execution readiness, supply chain capacity, and logistics practicality.
The Legal Worker assesses compliance, certifications, and intellectual property exposure.
The HR Worker evaluates hiring scale, role needs, and training readiness.

After all specialized reports are produced, the Manager Agent synthesizes them into one final strategic recommendation. This means the system does not stop at isolated analysis; it converts multi-domain intelligence into a practical launch decision.

This architecture demonstrates dynamic delegation, role-specialized workers, structured reporting, and centralized strategy synthesis in a refined multi-agent workflow.
""")


[Manager Agent] Reviewing strategic objective and assigning worker tasks...

[Market Research Worker] Collecting competitor data, customer preferences, and demand forecasts...

[Finance Worker] Analyzing budget, ROI, and pricing strategy...

[Legal Worker] Reviewing compliance, intellectual property, and regional regulations...

[HR Worker] Assessing staffing needs and training requirements...

[Manager Agent] Consolidating worker reports into final strategic recommendation...

CORPORATE MARKET RESEARCH & STRATEGY OUTPUT

1. Manager Planning Output
{
  "goal_summary": "Evaluate the feasibility of launching Product Y in Asia, focusing on establishing a strong presence in high-demand metro markets. The goal involves assessing market, financial, regulatory, and operational factors. Product Y targets urban, digitally active middle-income consumers in the consumer technology industry.",
  "assigned_workers": [
    "Market Research Worker",
    "Finance Worker",
    "Legal Worker",
    "HR W